# RLM vs RAG vs naive — интерактивный прогон

Запуск стенда из Jupyter с живым наблюдением за генерацией RLM.

**Перед запуском:**
1. Поднять модель (LM Studio или vLLM) и при необходимости указать конфиг.
2. Для RAG с локальным эмбеддером — скачать модель: `python -m scripts.download_embedder`.
3. Установить зависимости: `pip install -r requirements.txt` (нужен `ipywidgets`).

Конфиг: оставь поле `config` пустым (возьмётся `config.yaml` или `$RLM_CONFIG`) или укажи путь, напр. `config.vllm.yaml`.

In [ ]:
# Запускать из корня репозитория. Если ноутбук лежит в notebooks/ — поднимемся на уровень выше.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Рабочая папка:", os.getcwd())

In [ ]:
from eval.notebook_ui import launch

# Выбери параметры в форме и нажми «Запустить».
# quick=True — быстрый прогон (1 задача/тип, 1 длина) для проверки связи.
ui = launch(config_path="", dataset="simple", quick=True)

## Просмотр результатов

После прогона результаты лежат в `results.json` / `results.xlsx` (или `results_complex.*`). Сводку можно глянуть прямо тут.

In [ ]:
import json
import pandas as pd

path = "results.json"  # или results_complex.json
recs = json.load(open(path, encoding="utf-8"))["records"]
df = pd.DataFrame(recs)
df["tokens"] = df["usage"].apply(lambda u: u.get("total_tokens", 0))
summary = df.groupby(["method", "type"]).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
    avg_sec=("elapsed", "mean"),
    avg_tokens=("tokens", "mean"),
).round(2)
summary